In [1]:
from pypdf import PdfReader
import os 

# Load Resume + Job Descriptions

In [2]:
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    return text

def load_jobs(folder_path):
    jobs = []

    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                jobs.append(f.read())

    return jobs

# TEST LOADING

In [3]:
resume = load_pdf("Venkatesh_Vaishnav_+918177880563.pdf")
jobs = load_jobs("JDS")

print("Length of Resume", len(resume))
print("Number of Jobs", len(jobs))

Length of Resume 3279
Number of Jobs 1


# Create Embeddings

In [4]:
from sentence_transformers import SentenceTransformer

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    return model.encode(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Embed Jobs

In [6]:
job_embeddings = embed_texts(jobs)

print("Job Embedding Created:", len(job_embeddings))

Job Embedding Created: 1


# FAISS Search

In [7]:
import faiss
import numpy as np

In [8]:
dimension = job_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(job_embeddings))

print("FAISS Index Is Ready")

FAISS Index Is Ready


# Search

In [9]:
resume_embedding = embed_texts([resume])[0]

k = 1
distances, indices = index.search(np.array([resume_embedding]), k)

print("Top matches:\n")

for i in indices[0]:
    print(jobs[i][:300])
    print("=" * 50)

Top matches:

**JOB DESCRIPTION – MACHINE LEARNING ENGINEER (ENTRY LEVEL)**

**Job Title:** Machine Learning Engineer
**Experience:** 0–1 Years (Freshers with Internship Experience Preferred)
**Location:** India (On-site / Hybrid / Remote)

---

### **Role Overview**

We are looking for a passionate and detail-or


In [16]:
import json
import re

def clean_llm_output(output):
    try:
        json_part = re.search(r"\{.*\}", output, re.DOTALL).group()
        return json.loads(json_part)
    except:
        print("Error parsing:", output)
        return None

# Add LLM

In [17]:

import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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" 

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",  
    temperature=0
)

response = llm.invoke("Say hello like an AI engineer")
print(response.content)

Greetings. I'm online and ready to assist. How may I process your query today?


# Matching Function

In [18]:
def match_resume_to_job(resume, job_desc):
    prompt = f"""
    You are an AI hiring assistant.

    Compare the resume and job description.

    Resume:
    {resume}

    Job Description:
    {job_desc}

    IMPORTANT:
    - Return ONLY valid JSON
    - DO NOT add explanation
    - match_score must be an INTEGER between 0 and 100
    - DO NOT use decimal values like 0.85

    Format:
    {{
        "match_score": number,
        "missing_skills": [],
        "strengths": [],
        "suggestions": []
    }}
    """

    response = llm.invoke(prompt)
    return response.content

# Run AI Matching

In [20]:
for i in indices[0]:
    job = jobs[i]

    raw = match_resume_to_job(resume, job)
    result = clean_llm_output(raw)

    if result:
        # Fix score if needed
        if result["match_score"] <= 1:
            result["match_score"] *= 100

        print(f"""
================ MATCH RESULT ================

🎯 Match Score: {int(result["match_score"])}%

❌ Missing Skills:
{chr(10).join(["- " + skill for skill in result["missing_skills"]])}

💪 Strengths:
{chr(10).join(["- " + s for s in result["strengths"]])}

📌 Suggestions:
{chr(10).join(["- " + sug for sug in result["suggestions"]])}

=============================================
""")

    print("\n")  # extra spacing between results


================ MATCH RESULT ================

🎯 Match Score: 85%

❌ Missing Skills:
- SQL
- Git/GitHub

💪 Strengths:
- Strong foundation in Python
- Hands-on experience in working with real-world datasets
- Experience with ARIMA and SARIMA forecasting models
- Ability to automate data pipelines and improve pipeline efficiency

📌 Suggestions:
- Consider taking courses or gaining experience in Git/GitHub and SQL
- Highlight more experience in working with cross-functional teams and collaborating with data analysts and engineers
- Emphasize more on the ability to learn quickly and adapt to new technologies




